# Neptune Analytics Instance Management With S3 Table and S3 Vector Embedding Projections
This notebook demonstrates an end-to-end workflow where vector embeddings stored in Amazon S3 Vector are joined with data lake tabular data (an Iceberg table in S3 Tables), then imported into Amazon Neptune Analytics to perform a TopK similarity search.


### Prerequisite

To enable querying S3 Vector from Athena, the Athena S3 Vector Connector must be deployed in your account.

Installation instructions are available here:

../athena-s3vector-connector/README.md


### What this notebook covers:
1. Download the [Kaggle fashion dataset](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-dataset?resource=download), generate embeddings from item attributes using Amazon Bedrock, and write the embeddings to Amazon S3 Vector

2. Upload the original Kaggle dataset to simulate a source dataset from an enterprise data lake
3. Use Amazon Athena to create a projection that joins the S3 Tables dataset with S3 Vector embeddings to enrich the data
4. Import the enriched projection into Amazon Neptune Analytics
5. Run `topK.byNode` to retrieve similar products and return similarity scores


## Setup

Import the necessary libraries and set up logging.

In [ ]:
# Check the Python version:
from sys import version_info
assert version_info >= (3, 11), "Python 3.11 or higher is required"

import pandas as pd

from nx_neptune import empty_s3_bucket, instance_management, NeptuneGraph, set_config_graph_id
from nx_neptune.instance_management import execute_athena_query, _clean_s3_path
from nx_neptune.utils.utils import get_stdout_logger, validate_and_get_env, read_csv, \
    push_to_s3, to_embedding_entries, push_to_s3_vector, generate_create_table_ddl, generate_projection_stmt

# Configure logging to see detailed information about the instance creation process
logger = get_stdout_logger(__name__, [
    'nx_neptune.instance_management',
    'nx_neptune.utils.task_future',
    'nx_neptune.session_manager',
    'nx_neptune.interface',
    __name__
])

## Configuration

Check for environment variables necessary for the notebook.

In [ ]:
# Check for optional environment variables
dotenv.load_dotenv()
env_vars = validate_and_get_env([
    'NETWORKX_S3_DATA_LAKE_BUCKET_PATH',
    'NETWORKX_S3_NA_IMPORT_BUCKET_PATH',
    'NETWORKX_S3_LOG_BUCKET_PATH',
    'NETWORKX_S3_TABLES_DATABASE',
    'NETWORKX_S3_TABLES_TABLENAME',
    'S3_VECTOR_CONNECTOR',
    'S3_VECTOR_BUCKET',
    'S3_VECTOR_INDEX',
    'NETWORKX_GRAPH_ID'
])

(s3_location_data_lake, s3_location_na_import, s3_location_log, 
 s3_tables_database, s3_tables_tablename, s3_vector_connector,
 s3_vector_bucket, s3_vector_index, graph_id) = env_vars.values()


## Test Data Setup

The fashion product dataset used in this demo is sourced from Kaggle: [kaggle](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small).

For this workflow, only the styles.csv file is required.

In this section, we prepare two parallel data paths to simulate a realistic enterprise architecture:

1. **Structured Data Lake Source**

   The original fashion dataset is uploaded to Amazon S3 to simulate a typical Iceberg-backed table in an enterprise data lake.

2. **Embedding Generation Path**

    A subset of rows is extracted from the dataset to generate embedding vectors. These embeddings are then stored in Amazon S3 Vector, simulating a vector enrichment workflow.

This separation reflects a common production pattern where:
* Structured business data resides in the data lake
* Embeddings are generated asynchronously
* Vector data is stored independently and later joined during query time


In [ ]:
# Download the styles.csv from Kaggle dataset (Only the style.csv).
# https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
data_path = "../example/resources/styles.csv"

# Read data from data path
_, rows = read_csv(data_path)

# Inspect test data content.
df = pd.DataFrame(rows)
# To show the table on notebook.
df

### Test Data Upload – Embeddings

In this step, each selected row from the dataset is transformed to generate an embedding vector based on a subset of relevant product attributes.

The embedding is generated using Amazon Bedrock, then converted into a format compatible with Amazon S3 Vector.

The resulting vector entries are subsequently uploaded to the S3 Vector service, where they can later be queried and joined with structured data during Athena projection.


In [ ]:
# Add the embedding
columns_to_embed = ["masterCategory", "subCategory", "articleType",
                     "baseColour", "season", "year", "usage", "productDisplayName"]
embedding_limit = 1000

# Only produce embedding for first n items
items = to_embedding_entries(rows[:embedding_limit], columns_to_embed)

# Writing embedding S3 vector
push_to_s3_vector(items, s3_vector_bucket, s3_vector_index)

### Test Data Upload – Business Dataset (Data Lake Simulation)

In this step, the original styles.csv file is uploaded to Amazon S3 to simulate structured business data stored in a data lake.

An external table is then created in Amazon Athena using the defined schema, making the dataset queryable via SQL.

After this completes, the business data is ready to be joined with embedding data for enrichment.

In [ ]:
# Push to s3
empty_s3_bucket(s3_location_data_lake)
push_to_s3(data_path, _clean_s3_path(s3_location_data_lake), "styles.csv")

# Create the table representation on Athena with selected set of attributes
columns = [
    ("id", "string"),
    ("gender", "string"),
    ("masterCategory", "string"),
    ("subCategory", "string"),
    ("articleType", "string"),
    ("baseColour", "string"),
    ("season", "string"),
    ("year", "int"),
    ("usage", "string"),
    ("productDisplayName", "string")
]
stmt_s3_table = generate_create_table_ddl(s3_tables_tablename, s3_location_data_lake, columns)

await execute_athena_query(stmt_s3_table, s3_location_log, database=s3_tables_database)


print("DataLake preparation completed.")

### Data Enrichment and Graph Import

In this step, Amazon Athena is used to join the structured data lake table with embeddings stored in Amazon S3 Vector.

The enriched projection (including IDs, attributes, and embeddings) is written to S3 in a CSV format compatible with Amazon Neptune Analytics import requirements, cleaned, and then imported into Neptune Analytics.

After completion, the graph contains nodes enriched with embedding vectors, ready for similarity search.


In [ ]:
# Option 1 (Full table scan) - Generate SQL statement to join tabuler data with embedding entries
s3_vector_table_ref=f'"lambda:{s3_vector_connector}"."{s3_vector_bucket}"."{s3_vector_index}"'
stmt_projection = generate_projection_stmt(
    col_id="t.id",
    col_label="t.masterCategory",
    col_embedding="v.embedding",
    columns=["t.gender", "t.subCategory", "t.articleType",
             "t.baseColour", "t.season", "t.year",
             "t.usage", "t.productDisplayName"],
    base_table="test_embedding_table as t",
    joins=[
        (f"{s3_vector_table_ref} v", "t.id = v.vector_id")])

In [ ]:
# Option 2 (Ondemand lookup via UDF)
stmt_projection = generate_projection_stmt(
    col_id="t.id",
    col_label="t.masterCategory",
    col_vector_id="t.id",
    columns=["t.gender", "t.subCategory", "t.articleType",
             "t.baseColour", "t.season", "t.year",
             "t.usage", "t.productDisplayName"],
    base_table="test_embedding_table as t",
    connector_name=s3_vector_connector, 
    vector_bucket=s3_vector_bucket,
    vector_index=s3_vector_index)

In [ ]:

# Clear import directory
empty_s3_bucket(s3_location_na_import)

# Execute projection request
await execute_athena_query(stmt_projection, s3_location_na_import, database=s3_tables_database,
                          polling_interval=15)

# Remove unnecessary .csv.metadata file generated by Athena. 
empty_s3_bucket(s3_location_na_import, file_extension=".csv.metadata")

task_id = await instance_management.import_csv_from_s3(
        NeptuneGraph.from_config(set_config_graph_id(graph_id)),
        s3_location_na_import)

### Inspect Embeddings

A simple query is used to inspect the imported embeddings by printing the first 5 floating-point values from each node’s embedding vector. 

This provides a quick sanity check to verify that the embedding data has been ingested and stored correctly before running similarity queries.

In [ ]:
config = set_config_graph_id(graph_id)
na_graph = NeptuneGraph.from_config(config)

SHOW_EMBEDDING_QUERY = """
    MATCH (n) 
    CALL neptune.algo.vectors.get(n) 
    YIELD embedding 
    WHERE embedding is not null
    RETURN n, embedding[0..5] as embedding_first_five
    limit 3
"""

all_nodes = na_graph.execute_call(SHOW_EMBEDDING_QUERY)
for n in all_nodes:
    print(n["n"]["~id"] + ": " + str(n["embedding_first_five"]))

### Similarity Search

You can now run `neptune.algo.vectors.topK.byNode` to perform similarity search using the imported embedding vectors.

The following query first retrieves the embedding of the node with ID 30805, then executes the Top-K algorithm to identify other products that share similar semantic characteristics. Unlike conventional exact-match search, this approach can surface related items even when their attributes do not match exactly.

This confirms that the embeddings have been successfully imported and integrated into Amazon Neptune Analytics, and they are fully usable for semantic similarity search.

In [ ]:
TOPK_QUERY = """
    MATCH (n) WHERE id(n) = '30805'
    CALL neptune.algo.vectors.topK.byNode(
      n, {topK: 5})
    YIELD node, score
    RETURN node, score
"""

all_nodes = na_graph.execute_call(TOPK_QUERY)
rows = [{"score": n.get("score"), **n["node"]["~properties"]} for n in all_nodes]

df = pd.DataFrame(rows)
df

## Conclusion

This notebook demonstrated the complete lifecycle of embedding vectors — from being stored and managed in Amazon S3 Vector, to being joined with structured data in the data lake, and ultimately imported into Amazon Neptune Analytics for similarity search.

By integrating embeddings sourced from S3 Vector directly into the graph, this approach enables scalable and explainable similarity queries using native graph algorithms such as TopK. This is especially valuable for recommendation, product discovery, and data enrichment scenarios, where vector similarity must work alongside structured properties and graph relationships rather than operate as an isolated retrieval layer.

In practice, this pattern provides a flexible foundation for building hybrid graph-and-vector analytics pipelines that integrate cleanly with existing data lake architectures.